## model_based_cf_train

In [1]:
# 1. 데이터셋 다운로드
!wget -q http://files.grouplens.org/datasets/movielens/ml-100k.zip -P /content/
!unzip -qo /content/ml-100k.zip -d /content/
print('다운로드 완료')

다운로드 완료


In [2]:
# 2. 라이브러리 임포트
import math
import random
import pickle
from collections import defaultdict
from pathlib import Path
import numpy as np

In [3]:
# 3. 함수
def read_ratings(path):
    rows = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            user_id, item_id, rating, timestamp = line.strip().split("\t")
            rows.append((int(user_id), int(item_id), float(rating), int(timestamp)))
    return rows

def read_items(path):
    titles = {}
    genres = {}
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            parts = line.rstrip("\n").split("|")
            iid = int(parts[0])
            titles[iid] = parts[1]
            genre_vec = np.array([int(x) for x in parts[5:24]], dtype=np.float32)
            norm = np.linalg.norm(genre_vec)
            genres[iid] = genre_vec / norm if norm > 0 else genre_vec
    return titles, genres

def split_three_way(base_rows, valid_ratio=0.1, seed=42):
    by_user = defaultdict(list)
    for row in base_rows:
        by_user[row[0]].append(row)
    inner_train_rows, valid_rows = [], []
    for user_rows in by_user.values():
        sorted_rows = sorted(user_rows, key=lambda x: x[3])
        n_valid = max(1, int(len(sorted_rows) * valid_ratio)) if len(sorted_rows) >= 5 else 0
        valid_rows.extend(sorted_rows[-n_valid:] if n_valid > 0 else [])
        inner_train_rows.extend(sorted_rows[:-n_valid] if n_valid > 0 else sorted_rows)
    return inner_train_rows, valid_rows

In [4]:
# 4. MFRecommender 클래스
class MFRecommender:
    def __init__(self, n_factors=20, lr=0.005, reg=0.02, n_epochs=30,
                 min_candidate_popularity=1, seed=42):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
        self.min_candidate_popularity = min_candidate_popularity
        self.seed = seed

    def fit(self, train_rows):
        self.users = sorted({u for u, _, _, _ in train_rows})
        self.items = sorted({i for _, i, _, _ in train_rows})
        self.user_to_idx = {u: idx for idx, u in enumerate(self.users)}
        self.item_to_idx = {i: idx for idx, i in enumerate(self.items)}
        n_users, n_items = len(self.users), len(self.items)

        self.user_rated_items = defaultdict(list)
        self.item_popularity  = defaultdict(int)
        rating_sum = 0.0
        for user_id, item_id, rating, _ in train_rows:
            self.user_rated_items[user_id].append((item_id, rating))
            self.item_popularity[item_id] += 1
            rating_sum += rating
        self.global_mean = rating_sum / len(train_rows) if train_rows else 3.0

        rng = np.random.default_rng(self.seed)
        self.U   = rng.normal(0, 0.01, (n_users, self.n_factors)).astype(np.float32)
        self.V   = rng.normal(0, 0.01, (n_items, self.n_factors)).astype(np.float32)
        self.b_u = np.zeros(n_users, dtype=np.float32)
        self.b_i = np.zeros(n_items, dtype=np.float32)

        train_list = [(self.user_to_idx[u], self.item_to_idx[i], r) for u, i, r, _ in train_rows]
        rng_shuffle = random.Random(self.seed)
        for epoch in range(self.n_epochs):
            rng_shuffle.shuffle(train_list)
            sq_err = 0.0
            for uidx, iidx, rating in train_list:
                pred = (self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                        + float(self.U[uidx] @ self.V[iidx]))
                err    = rating - pred
                sq_err += err * err
                v_old = self.V[iidx].copy()
                self.V[iidx]   += self.lr * (err * self.U[uidx] - self.reg * self.V[iidx])
                self.U[uidx]   += self.lr * (err * v_old        - self.reg * self.U[uidx])
                self.b_u[uidx] += self.lr * (err - self.reg * self.b_u[uidx])
                self.b_i[iidx] += self.lr * (err - self.reg * self.b_i[iidx])
            if (epoch + 1) % 5 == 0:
                print(f"  epoch {epoch+1:>2}/{self.n_epochs}  train RMSE={math.sqrt(sq_err/len(train_list)):.4f}")
        return self

    def predict(self, user_id, item_id):
        has_u = user_id in self.user_to_idx
        has_i = item_id in self.item_to_idx
        if not has_u and not has_i: return self._clip(self.global_mean)
        if not has_u: return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])
        if not has_i: return self._clip(self.global_mean + self.b_u[self.user_to_idx[user_id]])
        uidx, iidx = self.user_to_idx[user_id], self.item_to_idx[item_id]
        return self._clip(self.global_mean + self.b_u[uidx] + self.b_i[iidx]
                          + float(self.U[uidx] @ self.V[iidx]))

    def recommend(self, user_id, top_n=10):
        if user_id not in self.user_to_idx:
            return self.popular_items(top_n)
        seen = {iid for iid, _ in self.user_rated_items[user_id]}
        candidates = [iid for iid in self.items
                      if iid not in seen
                      and self.item_popularity.get(iid, 0) >= self.min_candidate_popularity]
        scored = [(iid, self.predict(user_id, iid)) for iid in candidates]
        scored.sort(key=lambda x: (x[1], self.item_popularity.get(x[0], 0)), reverse=True)
        return scored[:top_n]

    def popular_items(self, top_n=10):
        sorted_items = sorted(self.item_popularity.keys(),
            key=lambda iid: (self.item_popularity[iid], self.item_mean(iid)), reverse=True)
        return [(iid, self.item_mean(iid)) for iid in sorted_items[:top_n]]

    def item_mean(self, item_id):
        if item_id not in self.item_to_idx: return self.global_mean
        return self._clip(self.global_mean + self.b_i[self.item_to_idx[item_id]])

    @staticmethod
    def _clip(value): return min(5.0, max(1.0, float(value)))

In [5]:
# 5. 평가 함수
def rating_metrics(model, rows):
    sq_errors, abs_errors = [], []
    for user_id, item_id, rating, _ in rows:
        pred = model.predict(user_id, item_id)
        e = rating - pred
        sq_errors.append(e * e)
        abs_errors.append(abs(e))
    return {
        "rmse": math.sqrt(sum(sq_errors) / len(sq_errors)),
        "mae":  sum(abs_errors) / len(abs_errors),
    }

def tune_factors(inner_train_rows, valid_rows, factor_values, lr, reg, n_epochs, min_pop):
    results = []
    for n_factors in factor_values:
        print(f"  n_factors={n_factors}")
        model = MFRecommender(n_factors=n_factors, lr=lr, reg=reg,
                              n_epochs=n_epochs, min_candidate_popularity=min_pop).fit(inner_train_rows)
        m = rating_metrics(model, valid_rows)
        results.append((n_factors, m["rmse"], m["mae"]))
        print(f"    RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}")
    best = min(results, key=lambda x: x[1])
    return best, results

In [6]:
# 6. 설정
factor_values = [10, 20, 50, 100]
lr            = 0.005
reg           = 0.02
n_epochs      = 30
min_candidate_popularity = 1
data_dir = Path('/content/ml-100k')

In [7]:
# 7. 데이터 로드 및 분할
base_rows      = read_ratings(data_dir / 'ua.base')
titles, genres = read_items(data_dir / 'u.item')

inner_train_rows, valid_rows = split_three_way(base_rows)
print(f"ua.base: {len(base_rows):,}  |  inner_train: {len(inner_train_rows):,}  |  valid: {len(valid_rows):,}")

ua.base: 90,570  |  inner_train: 81,917  |  valid: 8,653


In [8]:
# 8. n_factors 튜닝
print("[튜닝] n_factors 탐색")
best, validation_results = tune_factors(
    inner_train_rows, valid_rows, factor_values, lr, reg, n_epochs, min_candidate_popularity
)
best_factors = best[0]
print(f"\n→ 선택: n_factors={best_factors}")

del valid_rows

[튜닝] n_factors 탐색
  n_factors=10
  epoch  5/30  train RMSE=0.9373
  epoch 10/30  train RMSE=0.9229
  epoch 15/30  train RMSE=0.9175
  epoch 20/30  train RMSE=0.9143
  epoch 25/30  train RMSE=0.9102
  epoch 30/30  train RMSE=0.8998
    RMSE=0.9794  MAE=0.7804
  n_factors=20
  epoch  5/30  train RMSE=0.9372
  epoch 10/30  train RMSE=0.9228
  epoch 15/30  train RMSE=0.9173
  epoch 20/30  train RMSE=0.9138
  epoch 25/30  train RMSE=0.9088
  epoch 30/30  train RMSE=0.8963
    RMSE=0.9796  MAE=0.7806
  n_factors=50
  epoch  5/30  train RMSE=0.9371
  epoch 10/30  train RMSE=0.9225
  epoch 15/30  train RMSE=0.9164
  epoch 20/30  train RMSE=0.9111
  epoch 25/30  train RMSE=0.8999
  epoch 30/30  train RMSE=0.8771
    RMSE=0.9723  MAE=0.7738
  n_factors=100
  epoch  5/30  train RMSE=0.9369
  epoch 10/30  train RMSE=0.9219
  epoch 15/30  train RMSE=0.9151
  epoch 20/30  train RMSE=0.9073
  epoch 25/30  train RMSE=0.8898
  epoch 30/30  train RMSE=0.8595
    RMSE=0.9692  MAE=0.7712

→ 선택: n_factors=

In [9]:
# 9. 최종 모델 학습
print(f"[최종 학습] ua.base 전체 ({len(base_rows):,} rows), n_factors={best_factors}")
final_model = MFRecommender(
    n_factors=best_factors, lr=lr, reg=reg,
    n_epochs=n_epochs, min_candidate_popularity=min_candidate_popularity,
).fit(base_rows)

[최종 학습] ua.base 전체 (90,570 rows), n_factors=100
  epoch  5/30  train RMSE=0.9395
  epoch 10/30  train RMSE=0.9250
  epoch 15/30  train RMSE=0.9185
  epoch 20/30  train RMSE=0.9093
  epoch 25/30  train RMSE=0.8879
  epoch 30/30  train RMSE=0.8556


In [10]:
# 10. model.pkl 저장 + Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

import os
save_dir = '/content/drive/MyDrive/mf_model'
os.makedirs(save_dir, exist_ok=True)

bundle = {
    "model":              final_model,
    "titles":             titles,
    "genres":             genres,
    "best_factors":       best_factors,
    "validation_results": validation_results,
}
save_path = f"{save_dir}/model.pkl"
with open(save_path, "wb") as f:
    pickle.dump(bundle, f)
print(f"저장 완료: {save_path}")

Mounted at /content/drive
저장 완료: /content/drive/MyDrive/mf_model/model.pkl
